In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import importlib
from pathlib import Path

import data_pipeline
importlib.reload(data_pipeline)
from data_pipeline import DataPipeline, load_data, train_test_split

## Load Data

In [ ]:
data_path = Path("data/melb_data.csv")
df = load_data(data_path)
df.head()

## Train Test Split

In [ ]:
feature_df = df.drop(columns=["Price"])
target = df["Price"].to_numpy()
X_train_raw, X_test_raw, y_train, y_test = train_test_split(feature_df, target, test_size=0.2, random_state=42)

train_df = X_train_raw.copy()
train_df["Price"] = y_train

pd.DataFrame(
    {
        "split": ["train", "test"],
        "rows": [len(X_train_raw), len(X_test_raw)],
        "target_rows": [len(y_train), len(y_test)],
    }
)

## Target Distribution Check

In [ ]:
target_distribution_check = pd.DataFrame(
    {
        "train": pd.Series(y_train).describe(percentiles=[0.05, 0.5, 0.95]),
        "test": pd.Series(y_test).describe(percentiles=[0.05, 0.5, 0.95]),
    }
).round(2)
target_distribution_check

## First Look

In [ ]:
pd.DataFrame(
    {
        "metric": ["rows", "columns", "duplicate_rows", "target"],
        "value": [train_df.shape[0], train_df.shape[1], train_df.duplicated().sum(), "Price"],
    }
)

In [ ]:
train_df.dtypes.astype(str).rename("dtype").to_frame()

Ghi chú: dataset có kích thước nhỏ, xử lý trực tiếp bằng pandas là đủ. Dữ liệu gồm cả numeric và categorical; `Price` không có missing value.

## Missing And Invalid Values

In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing_count": train_df.isna().sum(),
        "missing_pct": train_df.isna().mean().mul(100).round(2),
    }
).query("missing_count > 0").sort_values("missing_pct", ascending=False)
missing_summary

In [ ]:
invalid_summary = pd.DataFrame(
    [
        {"check": "BuildingArea <= 0", "rows": int((train_df["BuildingArea"] <= 0).sum())},
        {"check": "YearBuilt < 1800", "rows": int((train_df["YearBuilt"] < 1800).sum())},
        {"check": "Landsize == 0", "rows": int((train_df["Landsize"] == 0).sum())},
        {"check": "BuildingArea > 1000", "rows": int((train_df["BuildingArea"] > 1000).sum())},
        {"check": "Landsize > 10000", "rows": int((train_df["Landsize"] > 10000).sum())},
    ]
)
invalid_summary

Ghi chú: `BuildingArea` và `YearBuilt` là hai cột thiếu nhiều nhất. `Car` thiếu rất ít. `CouncilArea` là categorical nên giá trị thiếu được gom vào nhóm `Unknown`. `BuildingArea <= 0`, `YearBuilt < 1800`, và `Landsize == 0` được xem là giá trị không hợp lệ khi tạo feature.

## Numeric Distribution

In [ ]:
numeric_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance", "Landsize", "BuildingArea", "YearBuilt", "Propertycount"]
numeric_profile = train_df[numeric_columns].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]).T
numeric_profile["skew"] = train_df[numeric_columns].skew(numeric_only=True)
numeric_profile.round(2)

In [ ]:
plot_columns = ["Price", "Distance", "Landsize", "BuildingArea"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, column in zip(axes.ravel(), plot_columns):
    ax.hist(train_df[column].dropna(), bins=40)
    ax.set_title(column)
plt.tight_layout()

Ghi chú: `Price` lệch phải mạnh và có outlier lớn. `Distance` tập trung nhiều ở vùng gần trung tâm. `Landsize` và `BuildingArea` có outlier rất lớn, do đó median phù hợp hơn mean cho imputation. `YearBuilt` có missing value nhiều và một giá trị năm bất hợp lý, cần repair trước khi tạo `PropertyAge`.

## Rooms And Simple Relationships

In [ ]:
corr_columns = ["Price", "Rooms", "Bedroom2", "Bathroom", "Car", "Distance", "BuildingArea", "YearBuilt"]
train_df[corr_columns].corr(numeric_only=True)[["Price", "Rooms"]].round(3)

Ghi chú: `Rooms`, `Bedroom2`, và `Bathroom` cùng mô tả quy mô căn nhà. `Bedroom2` gần với `Rooms`; `OtherRooms = Rooms - Bathroom` được dùng như feature layout đơn giản.

## Categorical Summary

In [ ]:
categorical_columns = ["Type", "Method", "Regionname", "CouncilArea"]
categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [train_df[column].nunique(dropna=True) for column in categorical_columns],
        "missing_pct": [round(train_df[column].isna().mean() * 100, 2) for column in categorical_columns],
        "top_value": [train_df[column].mode(dropna=True).iloc[0] for column in categorical_columns],
    }
)
categorical_summary

In [ ]:
type_price = train_df.groupby("Type")["Price"].agg(["count", "median", "mean"]).round(0)
region_price = train_df.groupby("Regionname")["Price"].agg(["count", "median"]).sort_values("median", ascending=False).round(0)
type_price, region_price

Ghi chú: `Type` tách rõ house, townhouse và unit. `Method` có cardinality thấp nên one-hot encode trực tiếp. `Regionname` và `CouncilArea` giữ tín hiệu vị trí với số nhóm vừa phải; missing `CouncilArea` được giữ thành nhóm `Unknown`.

## Geographical View

In [ ]:
bottom_99 = train_df["Price"] < train_df["Price"].quantile(0.99)
geo_df = train_df.loc[bottom_99]
fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    geo_df["Longtitude"],
    geo_df["Lattitude"],
    c=geo_df["Price"],
    cmap="viridis",
    alpha=0.35,
    s=12,
)
ax.set_xlabel("Longtitude")
ax.set_ylabel("Lattitude")
ax.set_title("Melbourne Housing Prices, bottom 99%")
fig.colorbar(scatter, ax=ax, label="Price")
plt.tight_layout()

Ghi chú: `Lattitude` và `Longtitude` cung cấp tín hiệu không gian trực tiếp. Biểu đồ chỉ lọc top 1% outlier của `Price` để dễ quan sát, không thay đổi dữ liệu trong pipeline.

## Feature Extraction Preview

In [ ]:
feature_preview = train_df[["Rooms", "Bathroom", "BuildingArea", "Landsize", "YearBuilt", "Date"]].copy()
feature_preview["Date"] = pd.to_datetime(feature_preview["Date"], dayfirst=True, errors="coerce")
feature_preview["OtherRooms"] = (feature_preview["Rooms"] - feature_preview["Bathroom"]).clip(lower=0)
feature_preview["Landsize_zero_or_missing"] = (feature_preview["Landsize"].isna() | (feature_preview["Landsize"] <= 0)).astype(float)
feature_preview["BuildingArea_per_Room"] = feature_preview["BuildingArea"] / feature_preview["Rooms"].replace(0, np.nan)
feature_preview["BuildingCoverage"] = feature_preview["BuildingArea"] / feature_preview["Landsize"].replace(0, np.nan)
feature_preview["PropertyAge"] = feature_preview["Date"].dt.year - feature_preview["YearBuilt"]
feature_preview[["OtherRooms", "BuildingArea_per_Room", "BuildingCoverage", "PropertyAge", "Landsize_zero_or_missing"]].describe().T.round(2)

Ghi chú: các feature tạo thêm giữ ở mức đơn giản: chênh lệch phòng, diện tích xây dựng trên mỗi phòng, tỷ lệ phủ xây dựng trên đất, tuổi nhà tại năm bán và indicator cho `Landsize` không dùng được khi tính ratio.

## Cleaning Decisions

| Vấn đề | Quan sát EDA | Quyết định pipeline | Lưu ý |
|---|---|---|---|
| `BuildingArea` missing hoặc `<= 0` | Thiếu nhiều, có giá trị không hợp lệ | Chuyển invalid thành NaN, impute median theo `Type`, thêm `BuildingArea_missing` | Không drop row để giữ dữ liệu |
| `YearBuilt` missing hoặc `< 1800` | Thiếu nhiều, có một năm bất hợp lý | Chuyển invalid thành NaN, impute median theo `Type`, thêm `YearBuilt_missing` | Dùng để tạo `PropertyAge` |
| `Landsize == 0` | Gây lỗi khi tạo ratio | Chuyển `Landsize <= 0` thành NaN, thêm `Landsize_zero_or_missing` | Dùng median fallback sau feature extraction |
| `CouncilArea` missing | Missing categorical | Fill bằng `Unknown` trước one-hot | Không drop row |
| Cột định danh/cardinality cao | `Address`, `SellerG`, `Suburb` có nhiều giá trị | Drop khỏi model matrix | Giữ tín hiệu vị trí qua region/council/lat-long |
| Feature ratio | `BuildingCoverage` có thể tạo inf nếu mẫu số không hợp lệ | Replace inf thành NaN, fallback bằng median train | Không clip quantile để tránh thêm state không cần thiết |

## Preprocessing Pipeline Handoff

In [ ]:
pipeline = DataPipeline()
X_train = pipeline.fit_transform(X_train_raw)
X_test = pipeline.transform(X_test_raw)

metadata = {
    "feature_names": pipeline.feature_names,
    "target_name": "Price",
    "pipeline": pipeline,
    "imputation_strategy": "grouped_median_by_type",
    "scaling_method": "standardize",
    "encoded_categorical_columns": pipeline.categorical_columns,
    "encoded_columns": pipeline.encoded_columns,
    "engineered_features": ["PropertyAge", "OtherRooms", "BuildingArea_per_Room", "BuildingCoverage", "BuildingArea_missing", "YearBuilt_missing", "Landsize_zero_or_missing"],
}

pd.DataFrame(
    {
        "artifact": ["X_train", "X_test", "y_train", "y_test", "features"],
        "shape": [X_train.shape, X_test.shape, y_train.shape, y_test.shape, len(metadata["feature_names"])],
    }
)

## Feature Contract Validation

In [ ]:
expected_engineered_features = metadata["engineered_features"]
X_train_df = pd.DataFrame(X_train, columns=metadata["feature_names"])
X_test_df = pd.DataFrame(X_test, columns=metadata["feature_names"])

feature_contract_check = pd.DataFrame(
    {
        "feature": expected_engineered_features,
        "exists_in_output": [feature in metadata["feature_names"] for feature in expected_engineered_features],
        "train_has_no_nan": [
            bool(not X_train_df[feature].isna().any()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_has_no_nan": [
            bool(not X_test_df[feature].isna().any()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
        "train_is_finite": [
            bool(np.isfinite(X_train_df[feature]).all()) if feature in X_train_df.columns else False
            for feature in expected_engineered_features
        ],
        "test_is_finite": [
            bool(np.isfinite(X_test_df[feature]).all()) if feature in X_test_df.columns else False
            for feature in expected_engineered_features
        ],
    }
)

failed_feature_contract = feature_contract_check.loc[
    ~feature_contract_check.drop(columns=["feature"]).all(axis=1)
]
assert failed_feature_contract.empty, failed_feature_contract.to_string(index=False)
feature_contract_check

## Clean Data Validation

In [ ]:
clean_checks = pd.DataFrame(
    [
        {"check": "X_train has no NaN", "passed": bool(not np.isnan(X_train).any())},
        {"check": "X_test has no NaN", "passed": bool(not np.isnan(X_test).any())},
        {"check": "X_train is finite", "passed": bool(np.isfinite(X_train).all())},
        {"check": "X_test is finite", "passed": bool(np.isfinite(X_test).all())},
        {"check": "same feature count", "passed": bool(X_train.shape[1] == X_test.shape[1])},
        {"check": "metadata feature count", "passed": bool(len(metadata["feature_names"]) == X_train.shape[1])},
        {"check": "y_train has no NaN", "passed": bool(not np.isnan(y_train).any())},
        {"check": "y_test has no NaN", "passed": bool(not np.isnan(y_test).any())},
        {"check": "y_train is finite", "passed": bool(np.isfinite(y_train).all())},
        {"check": "y_test is finite", "passed": bool(np.isfinite(y_test).all())},
        {"check": "y_train is positive", "passed": bool((y_train > 0).all())},
        {"check": "y_test is positive", "passed": bool((y_test > 0).all())},
        {"check": "X_train/y_train aligned", "passed": bool(X_train.shape[0] == len(y_train))},
        {"check": "X_test/y_test aligned", "passed": bool(X_test.shape[0] == len(y_test))},
    ]
)
assert clean_checks["passed"].all()
clean_checks

In [ ]:
pd.DataFrame({"feature_name": metadata["feature_names"]})